In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

print("Using device:", device)


LOCAL_RUN_NAME = "mm_clip_base_noaug_frozen"

BASE_DIR = Path.cwd().parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"

SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
OUTPUT_DIR_BASE = BASE_DIR / "outputs" / "image_modeling" / LOCAL_RUN_NAME
MODEL_DIR_BASE = BASE_DIR / "data" / "models" / LOCAL_RUN_NAME
FIGURE_DIR_BASE = BASE_DIR / "figures" / LOCAL_RUN_NAME

OUTPUT_DIR_BASE.mkdir(parents=True, exist_ok=True)
MODEL_DIR_BASE.mkdir(parents=True, exist_ok=True)
FIGURE_DIR_BASE.mkdir(parents=True, exist_ok=True)

CLIP_ID = "openai/clip-vit-base-patch32"
BATCH_SIZE = 8
INITIAL_LR = 1e-5
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
WEIGHT_DECAY = 1e-4
ACCUM_STEPS = 2
FREEZE_CLIP = True
NUM_WORKERS = 0
MAX_TEXT_LEN = 77
MODEL_NAME = "CLIP_Multimodal_M3_Frozen_Resume_FusionReady"
RESUME = True

TRAIN_SPLIT_PATH = SPLIT_DIR / "train_split.csv"
VAL_SPLIT_PATH = SPLIT_DIR / "val_split.csv"
LABEL2ID_PATH = SPLIT_DIR / "label2id.json"

BEST_CHECKPOINT = MODEL_DIR_BASE / "best_model_clip.pt"
LAST_CHECKPOINT = MODEL_DIR_BASE / "last_checkpoint_clip.pt"

HISTORY_CSV = OUTPUT_DIR_BASE / "history.csv"
VAL_PRED_CSV = OUTPUT_DIR_BASE / "val_predictions.csv"
FUSION_TABLE_CSV = OUTPUT_DIR_BASE / "val_fusion_table.csv"
CLASSIFICATION_REPORT_CSV = OUTPUT_DIR_BASE / "classification_report.csv"
RUN_METADATA_JSON = OUTPUT_DIR_BASE / "run_metadata.json"

LOGITS_NPY = OUTPUT_DIR_BASE / "y_logits.npy"
PROBA_NPY = OUTPUT_DIR_BASE / "y_proba.npy"

CONFUSION_MATRIX_PNG = FIGURE_DIR_BASE / "confusion_matrix.png"
TRAIN_LOSS_PNG = FIGURE_DIR_BASE / "train_loss.png"
TRAIN_ACC_PNG = FIGURE_DIR_BASE / "training_accuracy.png"
TRAIN_F1_PNG = FIGURE_DIR_BASE / "training_macro_f1.png"

print("Output root:", OUTPUT_DIR_BASE)
print("Model root :", MODEL_DIR_BASE)
print("Figure root:", FIGURE_DIR_BASE)
print("Best checkpoint path:", BEST_CHECKPOINT)
print("Last checkpoint path:", LAST_CHECKPOINT)


train_df = pd.read_csv(TRAIN_SPLIT_PATH)
val_df = pd.read_csv(VAL_SPLIT_PATH)

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    raw_label2id = json.load(f)

label2id = {int(k): int(v) for k, v in raw_label2id.items()}
id2label = {v: k for k, v in label2id.items()}

processor = CLIPProcessor.from_pretrained(CLIP_ID)


class RakutenCLIPDataset(Dataset):
    def __init__(self, df, processor, label_mapping, image_dir, max_text_len=77):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.label_mapping = label_mapping
        self.image_dir = image_dir
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.image_dir / f"image_{row['imageid']}_product_{row['productid']}.jpg"
        image = Image.open(img_path).convert("RGB")

        designation = "" if pd.isna(row.get("designation", "")) else str(row.get("designation", ""))
        description = "" if pd.isna(row.get("description", "")) else str(row.get("description", ""))
        text = f"{designation} {description}".strip()

        enc = self.processor(
            text=[text],
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.label_mapping[int(row["prdtypecode"])], dtype=torch.long)
        item["imageid"] = int(row["imageid"])
        item["productid"] = int(row["productid"])
        item["prdtypecode_raw"] = int(row["prdtypecode"])

        return item


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "imageid": [x["imageid"] for x in batch],
        "productid": [x["productid"] for x in batch],
        "prdtypecode_raw": [x["prdtypecode_raw"] for x in batch]
    }


train_dataset = RakutenCLIPDataset(train_df, processor, label2id, IMAGE_DIR, MAX_TEXT_LEN)
val_dataset = RakutenCLIPDataset(val_df, processor, label2id, IMAGE_DIR, MAX_TEXT_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    collate_fn=collate_fn
)


class CLIPMultimodalClassifier(nn.Module):
    def __init__(self, clip_id, num_classes, freeze_clip=True):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(clip_id)

        if freeze_clip:
            for p in self.clip.parameters():
                p.requires_grad = False

        proj_dim = self.clip.config.projection_dim

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        image_features = outputs.image_embeds
        text_features = outputs.text_embeds

        image_features = image_features / image_features.norm(dim=-1, keepdim=True).clamp(min=1e-12)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True).clamp(min=1e-12)

        combined = torch.cat([image_features, text_features], dim=1)

        return self.classifier(combined)


model = CLIPMultimodalClassifier(
    clip_id=CLIP_ID,
    num_classes=len(label2id),
    freeze_clip=FREEZE_CLIP
).to(device)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=INITIAL_LR,
    weight_decay=WEIGHT_DECAY
)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

def plot_history(history_df):
    if len(history_df) == 0:
        return

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{MODEL_NAME} - Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(TRAIN_LOSS_PNG, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="Train Accuracy")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{MODEL_NAME} - Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(TRAIN_ACC_PNG, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_f1"], label="Train Macro F1")
    plt.plot(history_df["epoch"], history_df["val_f1"], label="Val Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title(f"{MODEL_NAME} - Macro F1")
    plt.legend()
    plt.tight_layout()
    plt.savefig(TRAIN_F1_PNG, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

def run_one_epoch(model, loader, criterion, optimizer=None, accum_steps=1):
    train_mode = optimizer is not None

    if train_mode:
        model.train()
        optimizer.zero_grad(set_to_none=True)
    else:
        model.eval()

    running_loss = 0.0
    all_preds = []
    all_true = []
    val_rows = []
    all_logits = []
    all_proba = []

    for step, batch in enumerate(tqdm(loader, leave=False), start=1):
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.set_grad_enabled(train_mode):
            logits = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)

            if train_mode:
                (loss / accum_steps).backward()

                if step % accum_steps == 0 or step == len(loader):
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * labels.size(0)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

        if not train_mode:
            batch_logits = logits.detach().cpu().numpy()
            batch_proba = probs.detach().cpu().numpy()

            all_logits.append(batch_logits)
            all_proba.append(batch_proba)

            for j in range(labels.size(0)):
                pred_label_id = int(preds[j].detach().cpu().item())
                true_label_id = int(labels[j].detach().cpu().item())

                row_dict = {
                    "imageid": int(batch["imageid"][j]),
                    "productid": int(batch["productid"][j]),
                    "true_label_id": true_label_id,
                    "pred_label_id": pred_label_id,
                    "true_prdtypecode": int(id2label[true_label_id]),
                    "pred_prdtypecode": int(id2label[pred_label_id]),
                    "confidence": float(batch_proba[j].max())
                }

                for c in range(batch_logits.shape[1]):
                    row_dict[f"logit_{c}"] = float(batch_logits[j, c])
                    row_dict[f"proba_{c}"] = float(batch_proba[j, c])

                val_rows.append(row_dict)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds)
    epoch_f1 = f1_score(all_true, all_preds, average="macro")

    if not train_mode:
        all_logits = np.concatenate(all_logits, axis=0)
        all_proba = np.concatenate(all_proba, axis=0)
    else:
        all_logits = None
        all_proba = None

    return epoch_loss, epoch_acc, epoch_f1, all_true, all_preds, val_rows, all_logits, all_proba


start_epoch = 1
history = []
best_f1 = -1.0
epochs_without_improvement = 0
resume_path = None

if RESUME:
    if LAST_CHECKPOINT.exists():
        resume_path = LAST_CHECKPOINT
    elif BEST_CHECKPOINT.exists():
        resume_path = BEST_CHECKPOINT

if resume_path is not None:
    print(f"Resuming from checkpoint: {resume_path}")

    ckpt = torch.load(resume_path, map_location=device)

    model.load_state_dict(ckpt["model_state_dict"])

    if "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    if "scheduler_state_dict" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    history = ckpt.get("history", [])
    best_f1 = float(ckpt.get("best_f1", -1.0))
    epochs_without_improvement = int(ckpt.get("epochs_without_improvement", 0))
    start_epoch = int(ckpt.get("epoch", 0)) + 1

    print(f"Loaded epoch: {ckpt.get('epoch', 0)}")
    print(f"Next epoch: {start_epoch}")
    print(f"Best val F1 so far: {best_f1:.4f}")
    print(f"Loaded history rows: {len(history)}")
else:
    print("No checkpoint found. Starting from scratch.")


test_batch = next(iter(train_loader))

with torch.no_grad():
    test_logits = model(
        pixel_values=test_batch["pixel_values"].to(device),
        input_ids=test_batch["input_ids"].to(device),
        attention_mask=test_batch["attention_mask"].to(device)
    )

print("Sanity check logits:", test_logits.shape)


start_time = time.time()

if start_epoch <= MAX_EPOCHS:
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_t0 = time.time()

        train_loss, train_acc, train_f1, _, _, _, _, _ = run_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            accum_steps=ACCUM_STEPS
        )

        val_loss, val_acc, val_f1, y_true, y_pred, val_rows, y_logits, y_proba = run_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            optimizer=None,
            accum_steps=1
        )

        scheduler.step(val_f1)

        epoch_minutes = (time.time() - epoch_t0) / 60.0
        current_lr = optimizer.param_groups[0]["lr"]

        improved = val_f1 > best_f1

        if improved:
            best_f1 = val_f1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": current_lr,
            "epoch_minutes": epoch_minutes
        }

        history.append(epoch_record)

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "best_f1": best_f1,
            "epochs_without_improvement": epochs_without_improvement,
            "run_name": LOCAL_RUN_NAME,
            "model_name": MODEL_NAME
        }

        torch.save(checkpoint, LAST_CHECKPOINT)

        if improved:
            torch.save(checkpoint, BEST_CHECKPOINT)

            pd.DataFrame(val_rows).to_csv(VAL_PRED_CSV, index=False)
            pd.DataFrame(val_rows).to_csv(FUSION_TABLE_CSV, index=False)
            np.save(LOGITS_NPY, y_logits)
            np.save(PROBA_NPY, y_proba)

            print(f"Best model saved at epoch {epoch} with val_f1={val_f1:.4f}")
        else:
            print(f"No improvement for {epochs_without_improvement} epoch(s)")

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | "
            f"best_f1={best_f1:.4f} | "
            f"lr={current_lr:.2e} | "
            f"time={epoch_minutes:.2f} min",
            flush=True
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break
else:
    print(f"Checkpoint epoch already reached/exceeded MAX_EPOCHS ({MAX_EPOCHS}).")
    print("Set MAX_EPOCHS to a larger number if you want to continue training.")


total_minutes = (time.time() - start_time) / 60.0

best_source = BEST_CHECKPOINT if BEST_CHECKPOINT.exists() else LAST_CHECKPOINT

print(f"Loading best source for final evaluation: {best_source}")

best_ckpt = torch.load(best_source, map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

best_val_loss, best_val_acc, best_val_f1, y_true, y_pred, val_rows, y_logits, y_proba = run_one_epoch(
    model=model,
    loader=val_loader,
    criterion=criterion,
    optimizer=None,
    accum_steps=1
)

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_CSV, index=False)

val_pred_df = pd.DataFrame(val_rows)
val_pred_df.to_csv(VAL_PRED_CSV, index=False)
val_pred_df.to_csv(FUSION_TABLE_CSV, index=False)

np.save(LOGITS_NPY, y_logits)
np.save(PROBA_NPY, y_proba)

report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(CLASSIFICATION_REPORT_CSV)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest")
plt.title(f"{MODEL_NAME} - Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PNG, dpi=200, bbox_inches="tight")
plt.show()
plt.close()

plot_history(history_df)

best_epoch_from_history = None

if len(history_df) > 0:
    best_epoch_from_history = int(history_df.sort_values("val_f1", ascending=False).iloc[0]["epoch"])

run_metadata = {
    "run_name": LOCAL_RUN_NAME,
    "model_name": MODEL_NAME,
    "device": str(device),
    "clip_id": CLIP_ID,
    "batch_size": BATCH_SIZE,
    "initial_lr": INITIAL_LR,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "weight_decay": WEIGHT_DECAY,
    "accum_steps": ACCUM_STEPS,
    "freeze_clip": FREEZE_CLIP,
    "num_classes": len(label2id),
    "max_text_len": MAX_TEXT_LEN,
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "best_val_accuracy": float(best_val_acc),
    "best_val_macro_f1": float(best_val_f1),
    "best_val_loss": float(best_val_loss),
    "best_epoch": best_epoch_from_history,
    "total_training_minutes_this_session": float(total_minutes),
    "resumed_from_checkpoint": resume_path is not None,
    "resume_path": str(resume_path) if resume_path is not None else None,
    "best_checkpoint": str(BEST_CHECKPOINT),
    "last_checkpoint": str(LAST_CHECKPOINT),
    "history_csv": str(HISTORY_CSV),
    "val_predictions_csv": str(VAL_PRED_CSV),
    "fusion_table_csv": str(FUSION_TABLE_CSV),
    "classification_report_csv": str(CLASSIFICATION_REPORT_CSV),
    "logits_npy": str(LOGITS_NPY),
    "proba_npy": str(PROBA_NPY),
    "confusion_matrix_png": str(CONFUSION_MATRIX_PNG),
    "train_loss_png": str(TRAIN_LOSS_PNG),
    "train_acc_png": str(TRAIN_ACC_PNG),
    "train_f1_png": str(TRAIN_F1_PNG)
}

with open(RUN_METADATA_JSON, "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2, ensure_ascii=False)

print("\nFinal validation results")
print(f"Best Val Accuracy: {best_val_acc:.4f}")
print(f"Best Val Macro F1: {best_val_f1:.4f}")
print(f"Best Epoch From History: {best_epoch_from_history}")
print(f"Training time this session: {total_minutes:.2f} min")
print(f"History: {HISTORY_CSV}")
print(f"Predictions: {VAL_PRED_CSV}")
print(f"Fusion table: {FUSION_TABLE_CSV}")
print(f"Logits: {LOGITS_NPY}")
print(f"Probabilities: {PROBA_NPY}")
print(f"Classification report: {CLASSIFICATION_REPORT_CSV}")
print(f"Metadata: {RUN_METADATA_JSON}")
print(f"Best checkpoint: {BEST_CHECKPOINT}")
print(f"Last checkpoint: {LAST_CHECKPOINT}")
print(f"Figures: {FIGURE_DIR_BASE}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# PATH
BASE_DIR = Path.cwd().parent.parent.parent.parent
RUN_NAME = "mm_camembert_clip_aug_gatedfusion_unfreeze"

HISTORY_CSV_PATH = BASE_DIR / "outputs" / RUN_NAME / "training_history.csv"

# LOAD
df = pd.read_csv(HISTORY_CSV_PATH)

print(df.head())

# =========================
# ACCURACY
# =========================
plt.figure(figsize=(10,6))
plt.plot(df["global_epoch"], df["train_acc"], label="Train Accuracy")
plt.plot(df["global_epoch"], df["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over Epochs")
plt.legend()
plt.grid(False)
plt.show()

# =========================
# F1 SCORE
# =========================
plt.figure(figsize=(10,6))
plt.plot(df["global_epoch"], df["train_macro_f1"], label="Train Macro F1")
plt.plot(df["global_epoch"], df["val_macro_f1"], label="Validation Macro F1")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Macro F1 over Epochs")
plt.legend()
plt.grid(False)
plt.show()

# =========================
# LOSS
# =========================
plt.figure(figsize=(10,6))
plt.plot(df["global_epoch"], df["train_loss"], label="Train Loss")
plt.plot(df["global_epoch"], df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()
plt.grid(False)
plt.show()

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

BASE_DIR = Path.cwd().parent.parent.parent.parent

IMAGE_RUN_NAME = "mm_camembert_clip_aug_gatedfusion_unfreeze"

SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
IMAGE_RUN_DIR = BASE_DIR / "outputs" / IMAGE_RUN_NAME
TEXT_RUN_DIR = BASE_DIR / "outputs"

VAL_SPLIT_PATH = SPLIT_DIR / "val_split.csv"
LABEL2ID_PATH = SPLIT_DIR / "label2id.json"

IMAGE_LOGITS_PATH = IMAGE_RUN_DIR / "best_val_logits.npy"
TEXT_LOGITS_PATH = TEXT_RUN_DIR / "y_logits_camembert_run2_epochs3.npy"

val_df = pd.read_csv(VAL_SPLIT_PATH)

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    raw_label2id = json.load(f)
    label2id = {int(k): int(v) for k, v in raw_label2id.items()}

y_true = val_df["prdtypecode"].map(label2id).to_numpy()

image_logits = np.load(IMAGE_LOGITS_PATH)
text_logits = np.load(TEXT_LOGITS_PATH)

print("image_logits shape:", image_logits.shape)
print("text_logits shape:", text_logits.shape)
print("y_true shape:", y_true.shape)

if len(image_logits) != len(text_logits) or len(image_logits) != len(y_true):
    raise ValueError("Logits ve y_true satır sayıları aynı değil.")

X = np.concatenate([image_logits, text_logits], axis=1)

X_meta_train, X_meta_val, y_meta_train, y_meta_val = train_test_split(
    X,
    y_true,
    test_size=0.3,
    random_state=42,
    stratify=y_true
)

meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        C=1.0
    ))
])

meta_model.fit(X_meta_train, y_meta_train)

y_pred = meta_model.predict(X_meta_val)
y_proba = meta_model.predict_proba(X_meta_val)

acc = accuracy_score(y_meta_val, y_pred)
macro_f1 = f1_score(y_meta_val, y_pred, average="macro")
weighted_f1 = f1_score(y_meta_val, y_pred, average="weighted")

print("STACKING LOGIT FUSION RESULTS")
print("Accuracy :", round(acc, 6))
print("Macro F1 :", round(macro_f1, 6))
print("Weighted F1 :", round(weighted_f1, 6))
print()
print(classification_report(y_meta_val, y_pred, digits=4))

OUTPUT_DIR = BASE_DIR / "outputs" / "clip_aug_late_fusion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / "X_meta_train.npy", X_meta_train)
np.save(OUTPUT_DIR / "X_meta_val.npy", X_meta_val)
np.save(OUTPUT_DIR / "y_meta_train.npy", y_meta_train)
np.save(OUTPUT_DIR / "y_meta_val.npy", y_meta_val)
np.save(OUTPUT_DIR / "y_pred.npy", y_pred)
np.save(OUTPUT_DIR / "y_proba.npy", y_proba)

report_df = pd.DataFrame(classification_report(
    y_meta_val,
    y_pred,
    output_dict=True,
    zero_division=0
)).transpose()
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")

pred_df = pd.DataFrame({
    "true_label_id": y_meta_val,
    "pred_label_id": y_pred,
    "max_proba": y_proba.max(axis=1)
})
pred_df.to_csv(OUTPUT_DIR / "meta_val_predictions.csv", index=False)

print("Saved to:", OUTPUT_DIR)